In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

In [10]:
# === GPU 사용 설정 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU Name: Tesla T4


In [11]:
num_classes = 10
input_shape = (3, 224, 224)

x_train = np.random.random((1000, 3, 224, 224)).astype(np.float32)
y_train = np.random.randint(num_classes, size=(1000,))

x_test = np.random.random((200, 3, 224, 224)).astype(np.float32)
y_test = np.random.randint(num_classes, size=(200,))

train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train))
test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [12]:
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])  # GlobalAveragePooling을 위해 마지막 두 레이어를 제거

In [13]:
class CustomResNet50(nn.Module):
    def __init__(self, num_classes):
        super(CustomResNet50, self).__init__()
        self.base_model = base_model
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(2048, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
        # self.softmax = nn.Softmax(dim=1) # Softmax 제거

    def forward(self, x):
        x = self.base_model(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        # x = self.softmax(x)
        return x

model = CustomResNet50(num_classes=num_classes).to(device)

In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [15]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 2.32713533192873
Epoch 2, Loss: 2.2384281903505325
Epoch 3, Loss: 2.0724915862083435
Epoch 4, Loss: 1.627001229673624
Epoch 5, Loss: 1.3046270869672298
Epoch 6, Loss: 0.9769950658082962
Epoch 7, Loss: 0.6982626328244805
Epoch 8, Loss: 0.5253203036263585
Epoch 9, Loss: 0.4317201832309365
Epoch 10, Loss: 0.3646161868236959


In [16]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        # === 중요: 데이터도 GPU로 이동 ===
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 9.50%
